# RAG pipline

1. source: https://www.youtube.com/watch?v=KHePGkeFn54&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv&index=9, https://www.youtube.com/watch?v=JxaC6Hrym6c&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv&index=10
2. Note: If I use openAI, I can know the price

# Instead of putting the all documents which is so big, we need to search the context using minsearch, re-formatted and then build the prompt and send it to llm

In [ ]:



# docs = """General Course-Related Questions
# #I just discovered the course. Can I still join?
# Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.

# edit on GitHub
# #Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
# You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

# edit on GitHub
# #What is the video/zoom link to the stream for the “Office Hours” or live/workshop sessions?
# The zoom link is only published to instructors/presenters/TAs.

# Students participate via YouTube Live and submit questions to Slido (link is pinned in the chat when live). The video URL should be posted in the announcements channel on Telegram &amp; Slack before it begins. You can also watch live on the DataTalksClub YouTube Channel.

# Don’t post questions in chat as they may be missed if the room is very active.

#  """

# # Using ChatPromptTemplate.from_template
# # prompt = ChatPromptTemplate.from_template(
# #     """You are a strict, helpful assistant. Answer the user's question based ONLY on the provided context. 
# # If the answer cannot be found in the context, say 'I cannot find the answer in the text.' 
# # Do not use external knowledge or make up facts.

# # Context:
# # {context}

# # Question: {question}"""
# # )

# prompt = ChatPromptTemplate.from_messages([
#     (
#         "system", 
#         "You are a strict, helpful assistant. Answer the user's question based ONLY on the provided context. "
#         "If the answer cannot be found in the context, say 'I cannot find the answer in the text.' "
#         "Do not use external knowledge or make up facts.\n\n"
#         "Context:\n{context}"
#     ),
#     ("human", "{question}")
# ])

# llm = ChatOllama(
#     model="gemma3:4b",
#     temperature=0.9,
#     base_url="http://localhost:11434",
# )

# chain = 

# Build RAG with the Question and Context (Context depeig on the question that comes from minsearch)

In [3]:
# get the documents that are coming depending on search engine 



#####################################################
# 1. Build the Prompt
######################################################
import requests
from minsearch import Index


def search(question, course="llm-zoomcamp", num_results=5):
    boost_dict = {"question": 2.0, "section": 0.5}
    filter_dict = {"course": course}

    return index.search(
        question,
        boost_dict=boost_dict,
        filter_dict=filter_dict,
        num_results=num_results
    )
# 1.1. Get your data set, documents (a list of dictionaries)
# get the available courses
# each doc has course, course_path, path, questions_count fields
# the path contians the post past, "https://datatalks.club/faq" + "/json/data-engineering-zoomcamp.json" is wher ethe doc answer/question is saved 
docs_url = "https://datatalks.club/faq/json/courses.json"
response = requests.get(docs_url)
courses_raw = response.json()

documents = []
url_prefix = "https://datatalks.club/faq"

for course in courses_raw:
    course_url = f"""{url_prefix}{course["path"]}"""

    course_response = requests.get(course_url)
    course_response.raise_for_status()
    # get a list of dictionaries from a specific course
    course_data = course_response.json()
    # extend the list
    documents.extend(course_data)

# 1.2. Get your data set, documents (a list of dictionaries)
index = Index(
    text_fields=["question", "section", "answer"], # fields that conatin useful information to get the answer (fields that are used to perfom search)
    keyword_fields=["course"], # this is used when I want to filter, for example select * from documents where section="Machine Learning for Classification" then I will ignore all the courses and I will use only that section,
)
# fit with the documents (json file that contains all the prepared data)
index.fit(documents)

# get search result
# question = "I just discovered the course. Can I join now?"
# search_results = search(question, course="llm-zoomcamp", num_results=5)
# search_results


# 1.3. Building the context (rearrange the search result), cleaning
# Each document becomes a block with the section, question, and answer. This format makes it easy for the LLM to read. We turned a list of dictionaries into one string. It's a small preprocessing step before we send the data to the LLM
def build_context(search_results):
    lines = []

    for doc in search_results:
        lines.append(doc["section"])
        lines.append("Q: " + doc["question"])
        lines.append("A: " + doc["answer"])
        lines.append("")

    return "\n".join(lines).strip()
# context = build_context(search_results)

INSTRUCTIONS = """
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."
"""

USER_PROMPT_TEMPLATE = """
Question:
{question}

Context:
{context}
"""

def build_prompt(question, search_results):
    # rearrage the search_results so that llm model can read it easily (I think this is for each model, we need to build one)
    context = build_context(search_results)
    # the prompt needs to have question and answers
    prompt = USER_PROMPT_TEMPLATE.format(
        question=question,
        context=context
    )
    return prompt.strip()

# try it

question = "I just discovered the course. Can I join now?"

search_results = search(question, course="llm-zoomcamp", num_results=5)
# get re-formated prompt
prompt = build_prompt(question, search_results)
print(f"The prompt is: {prompt}")
print('------------------------------------------------------------------------------------------')

#####################################################
# 2.  call the llm model
# Since we re-arranged the prompt, we do not need a prompt template then
######################################################
# from langchain_core.prompts import ChatPromptTemplate
from langchain_ollama import ChatOllama
from langchain_core.output_parsers import StrOutputParser
# from langchain_classic.chains.combine_documents import create_stuff_documents_chain

llm = ChatOllama(
    model="gemma3:4b",
    temperature=0.9,
    base_url="http://localhost:11434",
)

chain =  llm | StrOutputParser()
response =  chain.invoke(prompt)
print(response)

The prompt is: Question:
I just discovered the course. Can I join now?

Context:
General Course-Related Questions
Q: I just discovered the course. Can I still join?
A: Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.

General Course-Related Questions
Q: Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
A: You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

General Course-Related Questions
Q: Certificate: Can I follow the course in a self-paced mode and get a certificate?
A: No, you can only get a certificate if you finish the course with a "live" cohort.

We don't award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitti

[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.


# Use Prompt Template instead of passsing the Prompt to the chain without prompt pipe

In [11]:
from langchain_core.prompts import ChatPromptTemplate


search_results = search(question, course="llm-zoomcamp", num_results=5)
cleaned_search_results = build_context(search_results)


prompt_test = ChatPromptTemplate.from_messages([
    (
        "system", 
        f"{INSTRUCTIONS}"
        "Context:\n{context}"
    ),
    ("human", "{question}")
])

llm_test = ChatOllama(
    model="gemma3:4b",
    temperature=0.9,
    base_url="http://localhost:11434",
)

chain_test = prompt_test | llm_test | StrOutputParser()
chain_test.invoke({  "context": cleaned_search_results,
     "question":  question
})

'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'

# Keeping the Conversation On, take into account  the history, not only answer indepent questions

1. sysytem:: the assistant’s behavior, ("system", "You are a helpful teaching assistant.")  
2. human is the user input, ("human", "{question}")
3. the assistance/model response, ("ai", "Yes, you can still join the course.")
4. developper is between sysytem and user (human)

SYSTEM:
You are a course assistant.

DEVELOPER:
Answer only from course FAQ.

USER:
Can I still join?

In [12]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage
from langchain_core.output_parsers import StrOutputParser
from langchain_ollama import ChatOllama


# 1. Instructions
INSTRUCTIONS = """
You are a helpful course assistant.

Answer the QUESTION based only on the CONTEXT.

If the answer is not in the context, say:
"I don't know."
"""

# 2. Chat history
chat_history = []


#3. Prompt
prompt_history  = ChatPromptTemplate.from_messages([
    (
        "system",
        INSTRUCTIONS
    ),
    MessagesPlaceholder(variable_name="chat_history"),
    (
        "human",
        """
Context:
{context}

Question:
{question}
"""
    )
])

# 4. LLM
llm = ChatOllama(
    model="gemma3:4b",
    temperature=0.9,
    base_url="http://localhost:11434",
)


# 4. Chain
# -----------------------------
chain_history = prompt_history  | llm | StrOutputParser()



# ONE chatbot function
def ask_question(question):
    
    # 1. Retrieve documents
    search_results = search(
        question,
        course="llm-zoomcamp",
        num_results=5
    )

    # 2. Build clean context
    cleaned_context = build_context(search_results)

    # 3. Invoke chain
    response = chain_history.invoke({
        "context": cleaned_context,
        "question": question,
        "chat_history": chat_history
    })

    # 4. Save memory
    chat_history.extend([
        HumanMessage(content=question),
        AIMessage(content=response)
    ])

    return response

response1 = ask_question(
    "Can I still join the course?"
)

print(response1)
response2 = ask_question(
    "Can I still get a certificate?"
)

print(response2)

Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.
Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.


# FINAL RAG  (Given the question, search the context, clean it, and get the answer, take also into account the previous conversion)
### Note1 this is without using retriver and without saving the context in a database (Chroma DB)
### Note2 we can also use from langchain_community.document_loaders to load documents that is stored in a specific folder
### Note3 RunnableLambda, RunnablePassthrough  if we want to add functions

In [ ]:
# llm
from langchain_ollama import ChatOllama 
from langchain_core.prompts import (
    ChatPromptTemplate,  # to set the prompt and specify the system, human, ai, and developper
    MessagesPlaceholder, # to keep the conversion on and take care from the previous conversion
)
from langchain_core.messages import (
    HumanMessage, # this is the user and we need to save the questions 
    AIMessage    # this the ai response, and we need to save it (also (save the anwsers from the AI))
)
from langchain_core.output_parsers import StrOutputParser  # set the output parser only string

# to get the docs and search a specific context
import requests
from minsearch import Index

# 0. Given a question, search from a list of dictionaries the good answer 




# 1.1. Get your data set, documents (a list of dictionaries)
# get the available courses
# each doc has course, course_path, path, questions_count fields
# the path contians the post past, "https://datatalks.club/faq" + "/json/data-engineering-zoomcamp.json" is wher ethe doc answer/question is saved 
docs_url = "https://datatalks.club/faq/json/courses.json"
response = requests.get(docs_url)
courses_raw = response.json()

documents = []
url_prefix = "https://datatalks.club/faq"

for course in courses_raw:
    course_url = f"""{url_prefix}{course["path"]}"""

    course_response = requests.get(course_url)
    course_response.raise_for_status()
    # get a list of dictionaries from a specific course
    course_data = course_response.json()
    # extend the list
    documents.extend(course_data)

# 1.2. Get your data set, documents (a list of dictionaries)
index = Index(
    text_fields=["question", "section", "answer"], # fields that conatin useful information to get the answer (fields that are used to perfom search)
    keyword_fields=["course"], # this is used when I want to filter, for example select * from documents where section="Machine Learning for Classification" then I will ignore all the courses and I will use only that section,
)
# fit with the documents (json file that contains all the prepared data)
index.fit(documents)

def search(question, course="llm-zoomcamp", num_results=5):
    boost_dict = {"question": 2.0, "section": 0.5}
    filter_dict = {"course": course}

    return index.search(
        question,
        boost_dict=boost_dict,
        filter_dict=filter_dict,
        num_results=num_results
    )


def build_context(search_results):
    lines = []

    for doc in search_results:
        lines.append(doc["section"])
        lines.append("Q: " + doc["question"])
        lines.append("A: " + doc["answer"])
        lines.append("")

    return "\n".join(lines).strip()

# 1. set the  instruction
INSTRUCTIONS = """
You are a helpful course assistant.

Answer questions ONLY using the provided context.

If the answer is not in the context, say:
"I don't know."
"""

# 2. set the history to be empty, chatbot
chat_history = []

# set the prompt
prompt = ChatPromptTemplate.from_messages([
    ("system",INSTRUCTIONS), # ai assistant
    MessagesPlaceholder(variable_name="chat_history"),  # takecare from the history

    # Current question + retrieved context (you need to put the question and also the clean context)
    ("human", """
Context:
{context}

Question:
{question}
""")
])







# Conversational RAG given question and a model name, this will be fetch
def rag(question, model="gemma3:4b"):
    global chat_history

    # given a question, use minisearch to get the first top search
    search_results = search(
            question,
            course="llm-zoomcamp", # which camp
            num_results=5
        )
    # clean the context to get better response
    context = build_context(search_results)

    answer = llm(model=model, question=question, context = context)
    return answer

question = "I just discovered the course. Can I join now?"
response = rag(question)
response


'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'

In [6]:
question2 = "How do I get a certificate?"

rag(question2)

'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'

In [7]:
rag("what do you mean, explain more")

"I don't know."